In [1]:
import os
import re
from pydub import AudioSegment
from TTS.api import TTS

text = """
Title: Sample narration.

This is a short test narration using XTTS v2 with a built in speaker. No cloning.
""".strip()

OUTPUT_FILE = "narration.mp3"
TMP_DIR = "tts_chunks"
MAX_CHARS = 700
SILENCE_MS = 250
BITRATE = "192k"

def split_into_chunks(text, max_chars=700):
    text = re.sub(r"\s+", " ", text).strip()
    parts = re.split(r"(?<=[\.\!\?])\s+", text)
    chunks = []
    cur = ""
    for p in parts:
        if not p:
            continue
        if len(cur) + len(p) + 1 <= max_chars:
            cur = (cur + " " + p).strip()
        else:
            if cur:
                chunks.append(cur)
            cur = p.strip()
    if cur:
        chunks.append(cur)
    return chunks

print("Loading model...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

speakers = tts.speakers
print("Speakers:", len(speakers))
print("First 10:", speakers[:10])

SPEAKER = speakers[0]

os.makedirs(TMP_DIR, exist_ok=True)
chunks = split_into_chunks(text, MAX_CHARS)

silence = AudioSegment.silent(duration=SILENCE_MS)
full = AudioSegment.empty()

for i, chunk in enumerate(chunks):
    wav_path = os.path.join(TMP_DIR, f"chunk_{i:04d}.wav")
    tts.tts_to_file(
        text=chunk,
        file_path=wav_path,
        language="en",
        speaker=SPEAKER,
    )
    seg = AudioSegment.from_wav(wav_path)
    full += seg + silence

full.export(OUTPUT_FILE, format="mp3", bitrate=BITRATE)
print("Done:", OUTPUT_FILE, "Duration(sec):", round(len(full) / 1000.0, 1))


c:\venv\tts\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading model...
Speakers: 58
First 10: ['Claribel Dervla', 'Daisy Studious', 'Gracie Wise', 'Tammie Ema', 'Alison Dietlinde', 'Ana Florence', 'Annmarie Nele', 'Asya Anara', 'Brenda Stern', 'Gitta Nikolina']
Done: narration.mp3 Duration(sec): 9.9


## narrator test

In [2]:
import os
from pydub import AudioSegment
from TTS.api import TTS

# ========= SETTINGS =========
AUDITION_TEXT = """
Today I will summarize a research paper in clear and neutral language.
I will explain the problem, the method, and the key results.
""".strip()

OUTPUT_DIR = "speaker_auditions"
FINAL_COMPARE_FILE = "speaker_compare.mp3"
MAX_SPEAKERS = 20
BITRATE = "192k"
SILENCE_MS = 700
# ============================

print("Loading XTTS v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

if not tts.speakers:
    raise RuntimeError("No speakers found in this model installation.")

# Sample speakers evenly if many exist
speakers = tts.speakers
if len(speakers) > MAX_SPEAKERS:
    step = max(1, len(speakers) // MAX_SPEAKERS)
    speakers = speakers[::step][:MAX_SPEAKERS]

print(f"Testing {len(speakers)} speakers...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

combined = AudioSegment.empty()
silence = AudioSegment.silent(duration=SILENCE_MS)

for idx, speaker in enumerate(speakers):
    wav_path = os.path.join(OUTPUT_DIR, f"{idx:02d}__{speaker}.wav")
    mp3_path = os.path.join(OUTPUT_DIR, f"{idx:02d}__{speaker}.mp3")

    print(f"Generating speaker {idx}: {speaker}")

    tts.tts_to_file(
        text=AUDITION_TEXT,
        file_path=wav_path,
        language="en",
        speaker=speaker,
    )

    seg = AudioSegment.from_wav(wav_path)
    seg.export(mp3_path, format="mp3", bitrate=BITRATE)

    combined += silence + seg

combined.export(FINAL_COMPARE_FILE, format="mp3", bitrate=BITRATE)

print("\nDone.")
print("Individual files saved in:", OUTPUT_DIR)
print("Combined comparison file:", FINAL_COMPARE_FILE)


Loading XTTS v2...
Testing 20 speakers...
Generating speaker 0: Claribel Dervla
Generating speaker 1: Gracie Wise
Generating speaker 2: Alison Dietlinde
Generating speaker 3: Annmarie Nele
Generating speaker 4: Brenda Stern
Generating speaker 5: Henriette Usha
Generating speaker 6: Tammy Grit
Generating speaker 7: Vjollca Johnnie
Generating speaker 8: Badr Odhiambo
Generating speaker 9: Royston Min
Generating speaker 10: Abrahan Mack
Generating speaker 11: Baldur Sanjin
Generating speaker 12: Damien Black
Generating speaker 13: Ilkin Urbano
Generating speaker 14: Ludvig Milivoj
Generating speaker 15: Torcull Diarmuid
Generating speaker 16: Zacharie Aimilios
Generating speaker 17: Maja Ruoho
Generating speaker 18: Lidiya Szekeres
Generating speaker 19: Szofi Granger

Done.
Individual files saved in: speaker_auditions
Combined comparison file: speaker_compare.mp3


In [ ]:
import os
from pydub import AudioSegment
from TTS.api import TTS

# ========= TEXT =========
text = """
Volatility is the central quantity in finance.
It governs how assets are priced, how portfolios are allocated, and how risk is managed.
And yet, for most of the history of quantitative finance, volatility has been treated as something hidden.
A latent variable that must be inferred indirectly from the behavior of returns.
""".strip()

# ========= SETTINGS =========
OUTPUT_FILE = "volatility_narration.mp3"
SPEAKER = "Torcull Diarmuid"
BITRATE = "192k"

# ============================
print("Loading XTTS v2...")
tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")

print("Using speaker:", SPEAKER)

wav_path = "temp_volatility.wav"

tts.tts_to_file(
    text=text,
    file_path=wav_path,
    language="en",
    speaker=SPEAKER,
)

AudioSegment.from_wav(wav_path).export(
    OUTPUT_FILE,
    format="mp3",
    bitrate=BITRATE
)

print("Done:", OUTPUT_FILE)


Loading XTTS v2...
Using speaker: Torcull Diarmuid
Done: volatility_narration.mp3


In [4]:
import os
from pydub import AudioSegment

# ========= CONFIG =========
START_INDEX = 20
END_INDEX = 50   # exclusive
OUTPUT_DIR = "speaker_range_test"
BITRATE = "192k"
LANGUAGE = "en"

TEXT = """
Volatility is the central quantity in finance.
""".strip()
# ===========================

if END_INDEX > len(tts.speakers):
    END_INDEX = len(tts.speakers)

os.makedirs(OUTPUT_DIR, exist_ok=True)

selected_speakers = tts.speakers[START_INDEX:END_INDEX]

print(f"Generating speakers {START_INDEX} to {END_INDEX-1}")
print(f"Total: {len(selected_speakers)} speakers\n")

for idx, speaker in enumerate(selected_speakers, start=START_INDEX):
    safe_name = speaker.replace(" ", "_").replace("/", "_")
    wav_path = os.path.join(OUTPUT_DIR, f"{idx:03d}__{safe_name}.wav")
    mp3_path = os.path.join(OUTPUT_DIR, f"{idx:03d}__{safe_name}.mp3")

    print(f"[{idx}] Generating: {speaker}")

    tts.tts_to_file(
        text=TEXT,
        file_path=wav_path,
        language=LANGUAGE,
        speaker=speaker,
    )

    AudioSegment.from_wav(wav_path).export(
        mp3_path,
        format="mp3",
        bitrate=BITRATE
    )

print("\nDone. Files saved in:", OUTPUT_DIR)


Generating speakers 20 to 49
Total: 30 speakers

[20] Generating: Abrahan Mack
[21] Generating: Adde Michal
[22] Generating: Baldur Sanjin
[23] Generating: Craig Gutsy
[24] Generating: Damien Black
[25] Generating: Gilberto Mathias
[26] Generating: Ilkin Urbano
[27] Generating: Kazuhiko Atallah
[28] Generating: Ludvig Milivoj
[29] Generating: Suad Qasim
[30] Generating: Torcull Diarmuid
[31] Generating: Viktor Menelaos
[32] Generating: Zacharie Aimilios
[33] Generating: Nova Hogarth
[34] Generating: Maja Ruoho
[35] Generating: Uta Obando
[36] Generating: Lidiya Szekeres
[37] Generating: Chandra MacFarland
[38] Generating: Szofi Granger
[39] Generating: Camilla Holmström
[40] Generating: Lilya Stainthorpe
[41] Generating: Zofija Kendrick
[42] Generating: Narelle Moon
[43] Generating: Barbora MacLean
[44] Generating: Alexandra Hisakawa
[45] Generating: Alma María
[46] Generating: Rosemary Okafor
[47] Generating: Ige Behringer
[48] Generating: Filip Traverse
[49] Generating: Damjan Chapma